# 00D — Test managed pipeline access without Docker or CDK

Run the cells in order using the current kernel and its AWS role. Only `boto3` is required.
This notebook creates a uniquely named SageMaker pipeline, updates it, then starts **one
small, billable Processing job** using an AWS-provided scikit-learn image. It uses two
synthetic values inside the container; no application code, model, input bucket or image
build is needed.

The run cell keeps one live summary on screen and saves full errors and job logs locally.
Capture its final output. Successful API submission and successful job execution are
reported separately. A failed operation is not automatically an IAM denial.

The optional last cell removes only this notebook's pipeline. It is **off by default**,
so you can first inspect the execution in Studio. Job history, CloudWatch logs and any
optional S3 output remain.


## Choose the execution settings

- Use the current AWS region and try to resolve the current assumed role into an execution role.
- Use the standard regional AWS image unless an approved image URI is supplied.
- Optionally supply an approved S3 output prefix, VPC settings and encryption keys.

Leave the blanks unchanged for the first attempt. If IAM role lookup is denied, the probe
tries the current role name without an IAM path and labels that assumption. Set
`execution_role_arn` explicitly if the platform uses a different role or an IAM path.
An API rejection will show which role and resource were actually submitted.

If your platform requires resource naming or tags, set `name_prefix` and `tags` before running.
No permissions, policies, domains, repositories or buckets are created or changed.


In [ ]:
from pathlib import Path
import json, re, textwrap, time, uuid
from urllib.parse import urlparse
import boto3
from botocore.config import Config
from IPython.display import clear_output

SETTINGS = {
    "region": "",                    # Empty uses the current AWS configuration.
    "execution_role_arn": "",         # Empty tries the current assumed role.
    "image_uri": "",                  # Empty uses the regional AWS scikit-learn image.
    "output_s3_uri": "",              # Optional approved s3://bucket/prefix/ for result.json.
    "instance_type": "ml.m5.large",
    "name_prefix": "managed-access",
    "tags": {},                      # Optional platform-required tags.
    "subnet_ids": [],
    "security_group_ids": [],
    "volume_kms_key_id": "",
    "output_kms_key_id": "",
    "wait_seconds": 900,
}

session = boto3.Session(region_name=SETTINGS["region"] or None)
client_config = Config(connect_timeout=5, read_timeout=30, retries={"max_attempts": 1})


## Define the small pipeline and its report

- Record each actual AWS call, retaining its full response or error locally.
- Put a short Python command directly in the Processing step, without downloading a script.
- Verify the updated revision and the expected result in CloudWatch or optional S3 output.

The AWS-owned image registry numbers below come from the
[SageMaker SDK image configuration](https://github.com/aws/sagemaker-python-sdk/blob/v2.253.1/src/sagemaker/image_uri_config/sklearn.json).
They are public service image accounts, not workload account settings. Other regions
can use an explicit approved `image_uri`.


In [ ]:
# Select the regional AWS image without installing the SageMaker SDK.
AWS_IMAGE_REGISTRIES = {
    "af-south-1": "510948584623",
    "ap-east-1": "651117190479",
    "ap-northeast-1": "354813040037",
    "ap-northeast-2": "366743142698",
    "ap-northeast-3": "867004704886",
    "ap-south-1": "720646828776",
    "ap-south-2": "628508329040",
    "ap-southeast-1": "121021644041",
    "ap-southeast-2": "783357654285",
    "ap-southeast-3": "951798379941",
    "ap-southeast-4": "106583098589",
    "ca-central-1": "341280168497",
    "ca-west-1": "190319476487",
    "eu-central-1": "492215442770",
    "eu-central-2": "680994064768",
    "eu-north-1": "662702820516",
    "eu-south-1": "978288397137",
    "eu-south-2": "104374241257",
    "eu-west-1": "141502667606",
    "eu-west-2": "764974769150",
    "eu-west-3": "659782779980",
    "il-central-1": "898809789911",
    "me-central-1": "272398656194",
    "me-south-1": "801668240914",
    "sa-east-1": "737474898029",
    "us-east-1": "683313688378",
    "us-east-2": "257758044811",
    "us-west-1": "746614075791",
    "us-west-2": "246618743249"
}

def client(service):
    """Take an AWS service name and return a client using the current credentials."""
    return session.client(service, config=client_config)

def show_report():
    """Save complete results locally and display one compact, updating summary."""
    WORK.mkdir(parents=True, exist_ok=True)
    (WORK / "details.json").write_text(json.dumps(REPORT, indent=2, default=str) + "\n")
    lines = ["MANAGED PIPELINE ACCESS — capture this output", ""]
    for key in ("account", "region", "role", "role_note", "image", "pipeline", "execution", "state"):
        if REPORT.get(key):
            lines.extend(textwrap.wrap(f"{key}: {REPORT[key]}", width=108,
                                       subsequent_indent="  ", break_on_hyphens=False))
    lines.append("")
    for name, result in REPORT["checks"].items():
        message = f"{result['status']:14} {name}"
        if result.get("detail"):
            message += " | " + " ".join(str(result["detail"]).split())
        lines.extend(textwrap.wrap(message, width=108, subsequent_indent="  ",
                                   break_on_hyphens=False))
    lines.extend(["", "Untested: Training, Batch Transform, Registry, custom-image push/pull.",
                  "S3 input reads are untested; S3 output is optional.",
                  "Not exercised means untested, not denied.",
                  f"Full errors and logs: {WORK.relative_to(ROOT)}/details.json"])
    if REPORT.get("pipeline_deleted"):
        lines.append("Pipeline deleted; job history, logs and any S3 output retained.")
    elif REPORT.get("pipeline_arn"):
        lines.append("Pipeline remains for inspection; removal is in the optional last cell.")
    else:
        lines.append("No pipeline creation was confirmed in this run.")
    summary = "\n".join(lines)
    (WORK / "summary.txt").write_text(summary + "\n")
    clear_output(wait=True)
    print(summary)

def record(name, status, detail=""):
    """Take an observed check result and refresh its saved and displayed summary."""
    REPORT["checks"][name] = {"status": status, "detail": detail}
    show_report()

def check(name, operation, *args, **arguments):
    """Run one operation and return its response, or record its exact failure."""
    try:
        response = operation(*args, **arguments)
    except Exception as error:
        payload = getattr(error, "response", {})
        code = payload.get("Error", {}).get("Code", type(error).__name__)
        denied = code in {"AccessDenied", "AccessDeniedException", "UnauthorizedOperation"}
        REPORT["errors"][name] = {"code": code, "message": str(error), "response": payload}
        record(name, "denied" if denied else "failed", str(error))
        return None
    REPORT["responses"][name] = response
    record(name, "confirmed")
    return response

def pipeline_definition(revision):
    """Take a revision number and return one Processing step that prints a verifiable result."""
    script = (
        "import json, pathlib;"
        f"result={{'probe':{TOKEN!r},'revision':{revision},'rows':2,'sum':sum([10,20])}};"
        "print('PROBE_EVENT_RECEIVED',flush=True);"
        "out=pathlib.Path('/opt/ml/processing/output');out.mkdir(parents=True,exist_ok=True);"
        "(out/'result.json').write_text(json.dumps(result));"
        "print('PROBE_RESULT '+json.dumps(result),flush=True);"
        "print('PROBE_SUCCESS',flush=True)"
    )
    cluster = {"InstanceCount": 1, "InstanceType": SETTINGS["instance_type"], "VolumeSizeInGB": 10}
    if SETTINGS["volume_kms_key_id"]:
        cluster["VolumeKmsKeyId"] = SETTINGS["volume_kms_key_id"]
    # Split the inline command to respect the API's 256-character limit per argument.
    arguments = {
        "ProcessingJobName": NAME + "-job",
        "RoleArn": REPORT["role"],
        "AppSpecification": {
            "ImageUri": REPORT["image"],
            "ContainerEntrypoint": ["python3", "-c", "import sys;exec(''.join(sys.argv[1:]))"],
            "ContainerArguments": [script[i:i + 256] for i in range(0, len(script), 256)],
        },
        "ProcessingResources": {"ClusterConfig": cluster},
        "StoppingCondition": {"MaxRuntimeInSeconds": 120},
    }
    if SETTINGS["subnet_ids"]:
        arguments["NetworkConfig"] = {"VpcConfig": {
            "Subnets": SETTINGS["subnet_ids"], "SecurityGroupIds": SETTINGS["security_group_ids"]}}
    if SETTINGS["output_s3_uri"]:
        REPORT["output_uri"] = SETTINGS["output_s3_uri"].rstrip("/") + "/" + NAME + "/"
        arguments["ProcessingOutputConfig"] = {"Outputs": [{
            "OutputName": "result", "S3Output": {"S3Uri": REPORT["output_uri"],
            "LocalPath": "/opt/ml/processing/output", "S3UploadMode": "EndOfJob"}}]}
        if SETTINGS["output_kms_key_id"]:
            arguments["ProcessingOutputConfig"]["KmsKeyId"] = SETTINGS["output_kms_key_id"]
    return {"Version": "2020-12-01", "Steps": [
        {"Name": "CheckAccess", "Type": "Processing", "Arguments": arguments}]}

def wait_for_execution(execution, timeout):
    """Poll an execution until it finishes or the observation timeout expires."""
    deadline = time.monotonic() + timeout
    while True:
        state = sm.describe_pipeline_execution(PipelineExecutionArn=execution)
        REPORT["state"] = state["PipelineExecutionStatus"]
        show_report()
        if REPORT["state"] not in {"Executing", "Stopping"}:
            return state
        if time.monotonic() >= deadline:
            raise TimeoutError("Observation timed out; the execution may still be active. Use Studio or cleanup.")
        time.sleep(10)

def read_job_logs(job_name):
    """Read this job's CloudWatch messages and briefly retry while its final logs arrive."""
    logs = client("logs")
    events = []
    deadline = time.monotonic() + 60
    for attempt in range(7):
        events = []
        try:
            pages = logs.get_paginator("filter_log_events").paginate(
                logGroupName="/aws/sagemaker/ProcessingJobs", logStreamNamePrefix=job_name + "/",
                startTime=REPORT["started_at"] - 60_000, endTime=int(time.time() * 1000))
            for page in pages:
                events.extend(page.get("events", []))
                REPORT["logs"] = events
                if any("PROBE_SUCCESS" in event["message"] for event in events):
                    return events
                if time.monotonic() >= deadline:
                    REPORT["log_read_note"] = "Log observation window ended; captured messages are retained."
                    return events
        except logs.exceptions.ResourceNotFoundException:
            if attempt == 6:
                raise
        REPORT["logs"] = events
        if any("PROBE_SUCCESS" in event["message"] for event in events) or attempt == 6:
            return events
        time.sleep(5)

def verify_result(result):
    """Take a result object and verify its probe identity, executed revision and arithmetic."""
    expected = {"probe": TOKEN, "revision": REPORT["expected_revision"], "rows": 2, "sum": 30}
    if result != expected:
        raise ValueError(f"Unexpected container result: {result}; expected {expected}")
    return result

def read_s3_result():
    """Read only this probe's optional S3 result and return its validated contents."""
    location = urlparse(REPORT["output_uri"] + "result.json")
    response = client("s3").get_object(Bucket=location.netloc, Key=location.path.lstrip("/"))
    with response["Body"] as body:
        return verify_result(json.load(body))

def run_probe():
    """Resolve the current role, create and update a pipeline, then run and inspect its Processing step."""
    identity = check("AWS caller", client("sts").get_caller_identity)
    if identity is None:
        return
    REPORT.update(account=identity["Account"], region=session.region_name)
    if not session.region_name:
        raise ValueError("Set region in SETTINGS or in the environment.")
    if not re.fullmatch(r"[a-z][a-z0-9-]{0,19}", SETTINGS["name_prefix"]):
        raise ValueError("Use a lowercase name_prefix of 1–20 letters, digits or hyphens.")
    if bool(SETTINGS["subnet_ids"]) != bool(SETTINGS["security_group_ids"]):
        raise ValueError("Provide both subnet_ids and security_group_ids, or leave both empty.")
    if SETTINGS["output_s3_uri"]:
        location = urlparse(SETTINGS["output_s3_uri"])
        if location.scheme != "s3" or not location.netloc or location.query or location.fragment:
            raise ValueError("output_s3_uri must be an approved s3://bucket/prefix/ URI.")
    REPORT["role"] = SETTINGS["execution_role_arn"]
    if not REPORT["role"]:
        arn = identity["Arn"]
        if ":assumed-role/" not in arn:
            raise ValueError("Set execution_role_arn to a role trusted by SageMaker.")
        role_name = arn.split(":assumed-role/", 1)[1].rsplit("/", 1)[0].split("/")[-1]
        role = check("Resolve current IAM role", client("iam").get_role, RoleName=role_name)
        if role is not None:
            REPORT["role"] = role["Role"]["Arn"]
        else:
            partition = arn.split(":")[1]
            REPORT["role"] = f"arn:{partition}:iam::{identity['Account']}:role/{role_name}"
            REPORT["role_note"] = "Derived from STS; IAM path unverified. Set execution_role_arn if needed."
    registry = AWS_IMAGE_REGISTRIES.get(session.region_name)
    REPORT["image"] = SETTINGS["image_uri"]
    if not REPORT["image"]:
        if not registry:
            raise ValueError("Set an approved image_uri for this region.")
        REPORT["image"] = f"{registry}.dkr.ecr.{session.region_name}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

    # Test create, update and start separately; an update denial does not block testing start.
    REPORT["pipeline"] = NAME
    tags = {**SETTINGS["tags"], "Purpose": "ManagedAccessProbe", "ProbeId": TOKEN}
    request = {"PipelineName": NAME, "RoleArn": REPORT["role"]}
    definition = pipeline_definition(1)
    (WORK / "definition.json").write_text(json.dumps(definition, indent=2) + "\n")
    created = check("CreatePipeline", sm.create_pipeline, **request,
                    PipelineDefinition=json.dumps(definition),
                    Tags=[{"Key": key, "Value": value} for key, value in tags.items()])
    if created is None:
        return
    REPORT["pipeline_arn"] = created["PipelineArn"]
    REPORT["expected_revision"] = 1
    updated = check("UpdatePipeline", sm.update_pipeline, **request,
                    PipelineDefinition=json.dumps(pipeline_definition(2)))
    if updated is not None:
        REPORT["expected_revision"] = 2
        (WORK / "definition.json").write_text(json.dumps(pipeline_definition(2), indent=2) + "\n")
    started = check("StartPipelineExecution", sm.start_pipeline_execution, PipelineName=NAME)
    if started is None:
        return
    REPORT["execution"] = started["PipelineExecutionArn"]

    # Observe service status and inspect the actual job, even when the execution fails.
    state = check("DescribePipelineExecution / wait", wait_for_execution,
                  REPORT["execution"], SETTINGS["wait_seconds"])
    if state is not None:
        status = state["PipelineExecutionStatus"]
        outcome = {"Succeeded": "confirmed", "Failed": "failed", "Stopped": "stopped"}.get(status, "pending")
        record("Execution result", outcome,
               status + (" | " + state["FailureReason"] if state.get("FailureReason") else ""))
    steps = check("ListPipelineExecutionSteps", sm.list_pipeline_execution_steps,
                  PipelineExecutionArn=REPORT["execution"])
    if steps is None:
        return
    for step in steps["PipelineExecutionSteps"]:
        if step.get("FailureReason"):
            record("Step failure", "failed", step["FailureReason"])
        job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
        if not job_arn:
            continue
        job_name = job_arn.rsplit("/", 1)[-1]
        job = check("DescribeProcessingJob", sm.describe_processing_job, ProcessingJobName=job_name)
        if job is not None:
            status = job["ProcessingJobStatus"]
            outcome = {"Completed": "confirmed", "Failed": "failed", "Stopped": "stopped"}.get(status, "pending")
            record("Processing result", outcome,
                   status + (" | " + job["FailureReason"] if job.get("FailureReason") else ""))
        events = check("Read CloudWatch logs", read_job_logs, job_name)
        if events is not None:
            messages = [event["message"] for event in events]
            results = [message.split("PROBE_RESULT ", 1)[1] for message in messages
                       if "PROBE_RESULT " in message]
            if results:
                result = check("Verify container output", verify_result, json.loads(results[-1]))
                if result is not None:
                    record("Verify container output", "confirmed",
                           f"revision={result['revision']}, rows={result['rows']}, sum={result['sum']}")
            else:
                record("Verify container output", "unverified", "No result marker found; inspect saved logs.")
            record("Entry and success logs",
                   "confirmed" if any("PROBE_EVENT_RECEIVED" in m for m in messages)
                   and any("PROBE_SUCCESS" in m for m in messages) else "unverified")
    if REPORT.get("output_uri"):
        check("Read and verify S3 output", read_s3_result)


## Create, update, execute and inspect

- Create a unique pipeline, change its embedded command from revision 1 to revision 2, and start it.
- Wait for the managed job and verify its result and entry/success messages in CloudWatch.
- Save the definition, full errors and logs locally; capture the single summary below.

This starts one instance with a 120-second processing runtime limit; provisioning can
take several minutes. The observation timeout does not delete or automatically stop
the execution. If the role is denied, the image is blocked, a quota is zero, or VPC/KMS
settings are required, the summary reports the actual failure.

After success, open **SageMaker Studio → Pipelines**, select the printed pipeline name,
and inspect its execution. A Processing success does not prove Training, Batch Transform,
Model Registry, custom-image ECR permissions or untested S3 access.


In [ ]:
TOKEN = uuid.uuid4().hex[:10]
NAME = SETTINGS["name_prefix"] + "-" + TOKEN
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mlops-models").is_dir()), Path.cwd())
WORK = ROOT / ".runs" / ("managed-access-" + TOKEN)
REPORT = {"checks": {name: {"status": "not exercised", "detail": ""} for name in (
    "CreatePipeline", "UpdatePipeline", "StartPipelineExecution",
    "DescribePipelineExecution / wait", "Execution result", "ListPipelineExecutionSteps",
    "DescribeProcessingJob", "Processing result", "Read CloudWatch logs",
    "Verify container output", "Entry and success logs", "Read and verify S3 output")},
    "responses": {}, "errors": {}, "logs": [], "started_at": int(time.time() * 1000)}
try:
    sm = client("sagemaker")
    run_probe()
except Exception as error:
    REPORT["errors"]["Probe setup"] = {"message": str(error), "type": type(error).__name__}
    REPORT["checks"]["Probe setup"] = {"status": "failed", "detail": str(error)}
finally:
    show_report()


## Optional: remove this probe's pipeline

- Leave the switch off to keep the pipeline available in Studio.
- When enabled, verify the current account and this run's ownership tag, then stop any active execution.
- Delete only the probe pipeline; retain job history, logs and optional S3 output.

Run this cell in the same kernel after inspecting the result. Cleanup permissions are
reported separately. If stopping or deletion is denied, the cell reports the remaining
pipeline instead of silently claiming it was removed.
Each run creates a new pipeline; use this cleanup cell before rerunning if you want to keep only one.


In [ ]:
DELETE_PIPELINE = False

def remove_probe_pipeline():
    """Verify this run's ownership, stop its active executions, and delete only its pipeline."""
    arn = REPORT.get("pipeline_arn")
    if not arn:
        return "No successfully created pipeline was recorded in this run."
    identity = client("sts").get_caller_identity()
    if identity["Account"] != REPORT["account"]:
        raise ValueError("Current credentials do not match the probe account.")
    tags = {tag["Key"]: tag["Value"] for tag in sm.list_tags(ResourceArn=arn)["Tags"]}
    if tags.get("ProbeId") != TOKEN or tags.get("Purpose") != "ManagedAccessProbe":
        raise ValueError("The pipeline does not have this probe's ownership tags.")
    for page in sm.get_paginator("list_pipeline_executions").paginate(PipelineName=NAME):
        for execution in page["PipelineExecutionSummaries"]:
            if execution["PipelineExecutionStatus"] in {"Executing", "Stopping"}:
                execution_arn = execution["PipelineExecutionArn"]
                if execution["PipelineExecutionStatus"] == "Executing":
                    sm.stop_pipeline_execution(PipelineExecutionArn=execution_arn)
                wait_for_execution(execution_arn, 300)
    sm.delete_pipeline(PipelineName=NAME)
    REPORT["pipeline_deleted"] = True
    return "Pipeline deleted; job history, logs and any S3 output retained."

if DELETE_PIPELINE:
    result = check("Optional pipeline cleanup", remove_probe_pipeline)
    if result is not None:
        record("Optional pipeline cleanup", "confirmed", result)
else:
    print("Cleanup is off. Set DELETE_PIPELINE = True here when ready.")
